# Optimizer Benchmark Analysis

Loads CSV I/II/III and Wandb run histories to produce paper-ready plots.

In [ ]:
import os
import warnings
warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
import seaborn as sns
import wandb

# ── Paths & constants ──────────────────────────────────────────────────────
RESULTS_DIR = "scripts/opt-bench/results"
CSV_I_PATH  = f"{RESULTS_DIR}/csv_i.csv"
CSV_II_PATH = f"{RESULTS_DIR}/csv_ii.csv"
CSV_III_PATH = f"{RESULTS_DIR}/csv_iii.csv"

WANDB_ENTITY  = "matanbt"
WANDB_PROJECT = "tropt-optbench"

# Paper-style aesthetics
sns.set_theme(style="whitegrid", font_scale=1.2)
PALETTE = "tab20"
FIG_DPI = 150

# Load CSVs
csv_i   = pd.read_csv(CSV_I_PATH)
csv_ii  = pd.read_csv(CSV_II_PATH)
csv_iii = pd.read_csv(CSV_III_PATH)

print(f"CSV I  rows: {len(csv_i)}")
print(f"CSV II rows: {len(csv_ii)}")
print(f"CSV III rows: {len(csv_iii)}")
print("Models:", csv_i["model_name"].unique().tolist())
print("Optimizers:", csv_i["optimizer_name"].unique().tolist())


## 1  Average Loss per Optimizer (CSV I)

Average final loss across all seeds and instructions.

In [ ]:
def bar_plot(df, value_col, title, ylabel, output_file=None,
             lower_bound_col=None, exclude_lower_bound=True):
    """Bar plot: avg value per optimizer, with std error bars."""
    plot_df = df.copy()
    if exclude_lower_bound and "is_lower_bound" in plot_df.columns:
        plot_df = plot_df[~plot_df["is_lower_bound"].astype(bool)]

    agg = (plot_df.groupby("optimizer_name")[value_col]
           .agg(["mean", "std", "count"])
           .reset_index()
           .sort_values("mean"))
    agg["se"] = agg["std"] / np.sqrt(agg["count"])

    fig, ax = plt.subplots(figsize=(max(8, len(agg) * 0.7), 5))
    colors = sns.color_palette(PALETTE, len(agg))
    bars = ax.bar(agg["optimizer_name"], agg["mean"], yerr=agg["se"],
                  capsize=4, color=colors, edgecolor="white")
    ax.set_xlabel("Optimizer")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    plt.xticks(rotation=35, ha="right")
    plt.tight_layout()
    if output_file:
        fig.savefig(output_file, dpi=FIG_DPI, bbox_inches="tight")
    plt.show()
    return agg

agg_loss = bar_plot(csv_i, "best_loss", "Average Final Loss per Optimizer", "Loss (↓ better)",
                    output_file=f"{RESULTS_DIR}/fig_avg_loss.pdf")
agg_loss


## 2  Average BLEU per Optimizer (CSV II)

In [ ]:
agg_bleu = bar_plot(csv_ii, "bleu", "Average BLEU Score per Optimizer", "BLEU (↑ better)",
                    output_file=f"{RESULTS_DIR}/fig_avg_bleu.pdf")
agg_bleu


## 3  Average Universality per Optimizer (CSV III)

Universality = mean `strongreject_finetuned` score across ClearHarm messages.

In [ ]:
agg_univ = bar_plot(csv_iii, "strongreject_finetuned",
                    "Average Universality Score per Optimizer",
                    "Universality / Jailbreakness (↑ = more harmful)",
                    output_file=f"{RESULTS_DIR}/fig_avg_universality.pdf")
agg_univ


## 4  Box Plots

First average across seeds per (optimizer, model, msg_id) to reduce seed noise, then plot distribution across messages/models.

In [ ]:
# Filter control — set to None to include all models
FILTER_MODELS = None  # e.g. ["google/gemma-2-2b-it"]

def box_plot(df, value_col, title, ylabel, output_file=None,
             exclude_lower_bound=True):
    plot_df = df.copy()
    if exclude_lower_bound and "is_lower_bound" in plot_df.columns:
        plot_df = plot_df[~plot_df["is_lower_bound"].astype(bool)]
    if FILTER_MODELS:
        plot_df = plot_df[plot_df["model_name"].isin(FILTER_MODELS)]

    # Average across seeds first
    seed_avg = (plot_df.groupby(["optimizer_name", "model_name", "msg_id"])[value_col]
                .mean().reset_index())

    fig, ax = plt.subplots(figsize=(max(8, seed_avg["optimizer_name"].nunique() * 0.9), 5))
    order = (seed_avg.groupby("optimizer_name")[value_col].median()
             .sort_values().index.tolist())
    sns.boxplot(data=seed_avg, x="optimizer_name", y=value_col, order=order,
                palette=PALETTE, ax=ax, width=0.6)
    ax.set_xlabel("Optimizer")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    plt.xticks(rotation=35, ha="right")
    plt.tight_layout()
    if output_file:
        fig.savefig(output_file, dpi=FIG_DPI, bbox_inches="tight")
    plt.show()

box_plot(csv_i,   "best_loss",              "Loss Distribution per Optimizer",         "Loss (↓ better)",
         f"{RESULTS_DIR}/fig_box_loss.pdf")
box_plot(csv_ii,  "bleu",                   "BLEU Distribution per Optimizer",          "BLEU (↑ better)",
         f"{RESULTS_DIR}/fig_box_bleu.pdf")
box_plot(csv_iii, "strongreject_finetuned", "Universality Distribution per Optimizer", "Universality (↑ = more harmful)",
         f"{RESULTS_DIR}/fig_box_universality.pdf")


## 5  Optimization Dynamics: FLOPs vs Loss

Line plot per optimizer for a chosen (model, message). Shading = std across seeds. Dashed line = soft_prompt lower bound.

In [ ]:
# Available message IDs in the data:
print("Available msg_ids:", sorted(csv_i["msg_id"].dropna().unique().tolist()))
print("Available models: ", csv_i["model_name"].unique().tolist())


In [ ]:
# ── Configuration ──────────────────────────────────────────────────────────
CHOSEN_MSG_ID = 0
CHOSEN_MODEL  = csv_i["model_name"].iloc[0]
X_AXIS = "step"   # "step" or "flops" (if flops are logged per step in wandb)

api = wandb.Api()
runs = api.runs(
    f"{WANDB_ENTITY}/{WANDB_PROJECT}",
    filters={
        "state": "finished",
        "config.run_type": "optbench_whitebox",
        "config.model_name": CHOSEN_MODEL,
        "config.msg_id": CHOSEN_MSG_ID,
    },
)

# Collect per-step histories grouped by (optimizer, seed)
histories: dict[str, dict[int, pd.DataFrame]] = {}
lb_histories: dict[int, pd.DataFrame] = {}   # soft_prompt lower bound

for run in runs:
    opt  = run.config.get("optimizer_name", "unknown")
    seed = run.config.get("seed", 0)
    is_lb = run.config.get("is_lower_bound", False)
    try:
        hist = run.history(keys=["loss", "step", "flops"], x_axis="_step", pandas=True)
    except Exception:
        hist = run.history(pandas=True)
    if hist.empty:
        continue
    # Normalise column names
    if "step" not in hist.columns and "_step" in hist.columns:
        hist = hist.rename(columns={"_step": "step"})
    if is_lb:
        lb_histories[seed] = hist
    else:
        histories.setdefault(opt, {})[seed] = hist

# ── Plot ───────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
colors = sns.color_palette(PALETTE, len(histories))

for color, (opt_name, seed_hists) in zip(colors, histories.items()):
    x_col = "flops" if (X_AXIS == "flops" and "flops" in next(iter(seed_hists.values())).columns) else "step"
    # Align on common x grid by interpolating
    all_x = sorted(set(x for h in seed_hists.values() for x in h[x_col].dropna()))
    interp_losses = []
    for hist in seed_hists.values():
        valid = hist[[x_col, "loss"]].dropna()
        if len(valid) < 2:
            continue
        interp_losses.append(np.interp(all_x, valid[x_col], valid["loss"]))
    if not interp_losses:
        continue
    arr = np.array(interp_losses)
    mean = arr.mean(axis=0)
    std  = arr.std(axis=0)
    ax.plot(all_x, mean, label=opt_name, color=color)
    ax.fill_between(all_x, mean - std, mean + std, alpha=0.15, color=color)

# Lower-bound dashed line
if lb_histories:
    x_col = "flops" if (X_AXIS == "flops" and "flops" in next(iter(lb_histories.values())).columns) else "step"
    lb_losses = []
    all_x_lb = sorted(set(x for h in lb_histories.values() for x in h[x_col].dropna()))
    for hist in lb_histories.values():
        valid = hist[[x_col, "loss"]].dropna()
        if len(valid) < 2:
            continue
        lb_losses.append(np.interp(all_x_lb, valid[x_col], valid["loss"]))
    if lb_losses:
        lb_mean = np.array(lb_losses).mean(axis=0)
        ax.plot(all_x_lb, lb_mean, linestyle="--", color="black", linewidth=1.5, label="soft_prompt (lower bound)")

ax.set_xlabel("FLOPs" if X_AXIS == "flops" else "Step")
ax.set_ylabel("Loss")
ax.set_title(f"Optimization Dynamics — model: {CHOSEN_MODEL.split('/')[-1]}, msg: {CHOSEN_MSG_ID}")
ax.legend(bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=9)
plt.tight_layout()
fig.savefig(f"{RESULTS_DIR}/fig_dynamics_m{CHOSEN_MSG_ID}.pdf", dpi=FIG_DPI, bbox_inches="tight")
plt.show()


## 6  LaTeX Table

Rows = optimizers, columns = models, cell = avg across seeds & instructions. Winners per column are marked in **bold**.

In [ ]:
METRIC_COL = "best_loss"  # swap to "bleu" or "strongreject_finetuned" for other metrics
METRIC_LOWER_IS_BETTER = True  # set False for BLEU / universality

df_tab = csv_i.copy()
if "is_lower_bound" in df_tab.columns:
    df_tab = df_tab[~df_tab["is_lower_bound"].astype(bool)]

# Average across seeds and msg_ids per (optimizer, model)
pivot = (df_tab.groupby(["optimizer_name", "model_name"])[METRIC_COL]
         .mean().unstack("model_name"))

# Add overall average column
pivot["Avg"] = pivot.mean(axis=1)
pivot = pivot.sort_values("Avg", ascending=METRIC_LOWER_IS_BETTER)

# Build LaTeX
model_cols = [c for c in pivot.columns if c != "Avg"]
header_row = " & ".join(["Optimizer"] + [c.split("/")[-1] for c in model_cols] + ["Avg"]) + r" \"

rows_latex = []
for opt, row in pivot.iterrows():
    vals = [row[c] for c in model_cols] + [row["Avg"]]
    # Find winner (min or max) per column
    formatted = []
    for col_idx, (col, v) in enumerate(zip(model_cols + ["Avg"], vals)):
        col_vals = pivot[col].dropna()
        is_winner = (v == col_vals.min()) if METRIC_LOWER_IS_BETTER else (v == col_vals.max())
        cell = f"\\textbf{{{v:.3f}}}" if is_winner else f"{v:.3f}"
        formatted.append(cell)
    rows_latex.append(f"    {opt} & " + " & ".join(formatted) + r" \\")

print(r"\begin{table}[h]")
print(r"\centering")
print(r"\caption{Optimizer Benchmark — " + METRIC_COL + "}")
print(r"\begin{tabular}{l" + "r" * (len(model_cols) + 1) + "}")
print(r"\toprule")
print(f"    {header_row}")
print(r"\midrule")
for r in rows_latex:
    print(r)
print(r"\bottomrule")
print(r"\end{tabular}")
print(r"\end{table}")
